In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import matplotlib.cm as cm
import datetime
import scipy.stats
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
%run ../20260323_read_data.py
df_sorted

In [ ]:
window_size1=12

fred = pd.read_excel('../20260309_freds-data/fred_age_model_v2.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['age_ma','d13c_ma','d18o_ma', 'd18o_pdb_ma']] = fred[['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()


cols = ['calendar_year','d13c','d18o','d18o_pdb']
new_cols = ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']
rename_dict = dict(zip(cols,new_cols))
right_roll = fred[cols].rolling(window_size1).mean().rename(columns=rename_dict)
left_roll = fred[cols].rolling(window_size1,center=True).mean().shift(window_size1).rename(columns=rename_dict)
fred.loc[:89, new_cols] = right_roll.loc[:89].to_numpy()
fred.loc[90:, new_cols] = left_roll.loc[90:].to_numpy()

fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[4:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['calendar_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred

In [ ]:
climate = pd.read_csv('../../../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate['datetime_month'] = climate['datetime'].dt.month
climate_monthly = climate[['SST','SSS','age_absolute', 'synthetic_d18o', 'synthetic_d18o_2','datetime_month']].groupby('datetime_month').mean()
climate_monthly

In [ ]:
tide = pd.read_csv('../../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

enso['Date'] = enso['Date'] + pd.offsets.MonthEnd(0)
enso.set_index('Date', inplace=True, drop=False)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
# ── ALIGNED: interpolate all external series to exact coral sample ages ─────
#
# WHY THE OLD APPROACH BROKE
# ---------------------------
# The previous code resampled coral to a 3ME grid, then joined ENSO / tide /
# climate via separate .reindex().interpolate().loc[target_dates] calls.
# Each join used a different intermediate index so after dropna() the surviving
# rows differed column by column — rolling().corr() then compared values at
# DIFFERENT calendar times, giving meaningless correlations.
#
# THE FIX
# -------
# Keep df_sorted at its exact irregular sample ages (decimal years).
# Map every external series onto those same ages with a single linear
# interpolator.  All columns share one index → rolling().corr() is correct.

from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

def interp_to(x_ref, y_ref, target):
    """Piecewise-linear interpolation; returns NaN outside x_ref range."""
    x_ref  = np.asarray(x_ref,  float)
    y_ref  = np.asarray(y_ref,  float)
    target = np.asarray(target, float)
    out    = np.full(len(target), np.nan)
    for i, a in enumerate(target):
        idx = np.searchsorted(x_ref, a)
        if 0 < idx < len(x_ref):
            t      = (a - x_ref[idx-1]) / (x_ref[idx] - x_ref[idx-1])
            out[i] = y_ref[idx-1] + t * (y_ref[idx] - y_ref[idx-1])
    return out

# Use exact sample ages as the shared axis
ages = df_sorted['age_final'].values

# ── Savitzky-Golay tectonic detrend (matches read_data.py) ────────────────────
# Build a monthly grid, apply SG (73-month window, order 3), interpolate back.
def to_monthly(a): return (np.array(a) * 12).round() / 12

df_sorted_copy = df_sorted.copy()
df_sorted_copy['mb'] = to_monthly(ages)
coral_mon = df_sorted_copy.groupby('mb')[['d13c', 'd18o_pdb']].mean().reset_index()
coral_mon.rename(columns={'mb': 'age'}, inplace=True)

for col in ['d13c', 'd18o_pdb']:
    sg = savgol_filter(
        coral_mon[col].interpolate().ffill().bfill().values,
        window_length=73, polyorder=3, mode='interp'
    )
    fn = interp1d(coral_mon['age'].values, sg,
                  bounds_error=False, fill_value='extrapolate')
    df_sorted[f'{col}_sg_trend']  = fn(ages)
    df_sorted[f'{col}_no_tec_sg'] = df_sorted[col] - df_sorted[f'{col}_sg_trend']

# ── Climate ────────────────────────────────────────────────────────────────────
clim_s = climate.sort_values('age_absolute')
for col in ['SST', 'SSS', 'synthetic_d18o']:
    df_sorted[col] = interp_to(clim_s['age_absolute'].values, clim_s[col].values, ages)

# ── Tide gauge ────────────────────────────────────────────────────────────────
tide_s  = tide.dropna(subset=['Sea Level']).sort_values('Date')
sl_raw  = interp_to(tide_s['Date'].values, tide_s['Sea Level'].values, ages)
df_sorted['Sea Level'] = (sl_raw - np.nanmean(sl_raw)) / 1000.0   # mm → m, remove mean

# ── ENSO ──────────────────────────────────────────────────────────────────────
enso_dt  = pd.to_datetime(enso['Date'])
enso_age = (enso_dt.dt.year + (enso_dt.dt.month - 0.5) / 12).values
order    = np.argsort(enso_age)
df_sorted['ENSO'] = interp_to(enso_age[order], enso['ENSO'].values[order], ages)

# ── Build aligned: single float index, one row per coral sample ──────────────
aligned = df_sorted[[
    'age_final',
    'd13c', 'd18o', 'd18o_pdb',
    'd13c_no_tec_sg', 'd18o_pdb_no_tec_sg',
    'SST', 'SSS', 'synthetic_d18o',
    'Sea Level', 'ENSO'
]].rename(columns={'age_final': 'age'}).set_index('age').sort_index()

print(f"aligned: {len(aligned)} rows  {aligned.index.min():.2f}–{aligned.index.max():.2f}")
print(f"  SST:       {aligned['SST'].notna().sum()} samples")
print(f"  Sea Level: {aligned['Sea Level'].notna().sum()} samples")
print(f"  ENSO:      {aligned['ENSO'].notna().sum()} samples")
aligned


In [ ]:
sampling_rates = df[~df['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()

In [ ]:
dt = np.diff(df_sorted['age_final'].values)
plt.hist(dt * 12, bins=50)  # in months
plt.xlabel('Sampling interval (months)')

In [ ]:
df['age_final']

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(sampling_rates, '.-')

ax = axes[1]

mu = np.mean(sampling_rates)
sigma = np.std(sampling_rates)
bin_count = 16

count, bins, ignored = ax.hist(sampling_rates, bins=np.arange(1,bin_count), rwidth=0.9, align='left',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='r')
ax.set_yticks(np.arange(1,bin_count+1))
ax.set_ylim(1,bin_count+1)


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Sampling Rate Distribution')
axes[0].set_title('Sampling Rate over Time')
axes[0].set_ylabel('# of Sample per Year')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')

#fig.savefig('sampling_rate_distribution.pdf')

In [ ]:

growth_rates = (sampling_rates*1.5)/10

# Drill Samples were 1.5 mm ?????

#####################################

fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(growth_rates, '.-')
ax.grid()

ax = axes[1]

mu = np.mean(growth_rates)
sigma = np.std(growth_rates)
bin_count = 22

count, bins, ignored = ax.hist(growth_rates, bins=np.arange(1,bin_count)/10, rwidth=0.7, align='left',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='r')
ax.set_yticks(np.arange(1,bin_count+1,2)/10)
ax.set_ylim(1/10,bin_count/10+1/10)
ax.grid()

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_ylabel("Growth Rate [cm/a]")


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Growth Rate Distribution')
axes[0].set_title('Growth Rate over Sample Age')
axes[0].set_ylabel('Growth Rate [cm/a]')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')


fig.savefig('sampling_rate_distribution.pdf')

In [ ]:
from astropy.timeseries import LombScargle
import numpy as np

# Use detrended signal
data = df_sorted[['age_final', 'd13c_no_tectonic']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic'])
frequency_notec , power_notec = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)

period_notec = 1 / frequency_notec



In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 7))

ax.plot(period_notec, power_notec)
ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.set_xlabel('Period (years)')
ax.set_xlim(0.5, 7)

ax.legend()

In [ ]:
df_sorted['d13c_no_tec_sg']

In [ ]:
# Use detrended signal
data = df_sorted[['age_final', 'd13c_no_tec_sg']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tec_sg'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)

period_notec2 = 1 / frequency_notec2

fig, ax = plt.subplots(1, figsize=(12, 7))

ax.plot(period_notec2, power_notec2)
ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.set_xlabel('Period (years)')
ax.set_xlim(0.5, 7)

ax.legend()
fig

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(12, 7))

ax = axes[0]
ax.plot(period_notec2, power_notec2)
ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.set_xlabel('Period (years)')
ax.set_xlim(0.5, 7)
ax.legend()

ax = axes[1]
ax.plot(period_notec2, power_notec2)
ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.axvline(0.935, color='g', linestyle='--', alpha=0.4,label='341 days')
ax.set_xlabel('Period (years)')
ax.set_xlim(0.05, 1.1 )
ax.legend()
fig

In [ ]:
# Use detrended signal
data = df_sorted[['age_final', 'd13c']].dropna()

ls = LombScargle(data['age_final'], data['d13c'])
frequency, power = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)

period = 1 / frequency



In [ ]:
data = df_sorted[['age_final', 'd13c']].dropna()

ls = LombScargle(data['age_final'], data['d13c'])
frequency, power = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period = 1 / frequency

data = df_sorted[['age_final', 'd13c_no_tectonic']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec = 1 / frequency_notec



data = df_sorted[['age_final', 'd13c_no_tectonic2']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic2'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec2 = 1 / frequency_notec2



In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))


greens = cm.get_cmap('Greens')

colors = [
    greens(0.35),  # light green (raw)
    greens(0.65),  # medium green (36m)
    greens(0.9)    # dark green (72m)
]

ax.plot(period, power,color=colors[0], label='raw')
ax.plot(period_notec, power_notec,color=colors[1], label='36 month filter')
ax.plot(period_notec2, power_notec2,color=colors[2], label='72 month filter')

ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.axvline(3.617, color='g', linestyle='--', alpha=0.4,label='3.6 year')
ax.set_xlabel('Period (years)')
ax.set_ylabel('Spectral Amplitude')
ax.set_title(r'$\delta^{13}$C Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0,0.04)

ax.legend(loc='upper right')

#fig.savefig('lombscargle_raw-filter_d13c.pdf')

In [ ]:
data = df_sorted[['age_final', 'd18o']].dropna()

ls = LombScargle(data['age_final'], data['d18o'])
frequency, power = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period = 1 / frequency

data = df_sorted[['age_final', 'd18o_no_tectonic']].dropna()

ls = LombScargle(data['age_final'], data['d18o_no_tectonic'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec = 1 / frequency_notec



data = df_sorted[['age_final', 'd18o_no_tectonic2']].dropna()

ls = LombScargle(data['age_final'], data['d18o_no_tectonic2'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec2 = 1 / frequency_notec2



In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))

blues = cm.get_cmap('Blues')

colors = [
    blues(0.35),  # light green (raw)
    blues(0.65),  # medium green (36m)
    blues(0.9)    # dark green (72m)
]

ax.plot(period, power,color=colors[0], label='raw')
ax.plot(period_notec, power_notec,color=colors[1], label='36 month filter')
ax.plot(period_notec2, power_notec2,color=colors[2], label='72 month filter')


ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.axvline(3.617, color='g', linestyle='--', alpha=0.4,label='3.6 year')
ax.set_xlabel('Period (years)')
ax.set_ylabel('Spectral Amplitude')
ax.set_title(r'$\delta^{13}$C Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0,0.15)

ax.legend(loc='upper right')

#fig.savefig('lombscargle_raw-filter_d18o.pdf')

In [ ]:
data = enso[['decimal_year', 'ENSO']].dropna()

ls = LombScargle(data['decimal_year'], data['ENSO'])
frequency_enso, power_enso = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_enso = 1 / frequency_enso

data = climate[['age_absolute', 'synthetic_d18o']].dropna()

ls = LombScargle(data['age_absolute'], data['synthetic_d18o'])
frequency_synd18o, power_synd18o = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_synd18o = 1 / frequency_synd18o



data = climate[['age_absolute', 'SSS']].dropna()

ls = LombScargle(data['age_absolute'], data['SSS'])
frequency_SSS, power_SSS = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_SSS = 1 / frequency_SSS



In [ ]:
fig, axes = plt.subplots(3,1, figsize=(8, 9), sharex=True, gridspec_kw={'hspace':0})

periods = [period_enso, period_synd18o, period_SSS]
powers = [power_enso, power_synd18o, power_SSS]
colors = ['#4F4948', '#0827FF', '#E30EDE']
labels = ['ENSO', r'Climate $\delta^{18}$O', 'SSS']


for i in range(3):
    
    ax = axes[i]
    ax.plot(periods[i], powers[i], colors[i], label=labels[i])
    ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
    ax.axvline(3.617, color='g', linestyle='--', alpha=0.4,label='3.6 year')
    ax.set_xlabel('Period (years)')
    ax.set_ylabel('Spectral Amplitude')
    #ax.set_title(r'$\delta^{13}$C Spectrum')
    ax.set_xlim(0.5, 7)
    ax.set_ylim(0,0.15)
    ax.legend(loc='upper right')

#axes[1].set_ylim(0, 0.2)
#axes[2].set_ylim(0, 0.2)
ax.legend(loc='upper right')

#fig.savefig('spectral_enso-d18o-sss.pdf')

In [ ]:
df_sorted

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astropy.timeseries import LombScargle

def lomb_spec(df, xcol, ycol, fmin=0.001, fmax=12, spp=10):
    d = df[[xcol, ycol]].dropna()
    ls = LombScargle(d[xcol].values, d[ycol].values)
    freq, power = ls.autopower(minimum_frequency=fmin,
                               maximum_frequency=fmax,
                               samples_per_peak=spp)
    period = 1.0 / freq
    return period, power

# ---- compute spectra ----
# d13c
p_d13c_raw,  pow_d13c_raw  = lomb_spec(df_sorted, 'age_final', 'd13c')
p_d13c_36,   pow_d13c_36   = lomb_spec(df_sorted, 'age_final', 'd13c_no_tectonic')
p_d13c_72,   pow_d13c_72   = lomb_spec(df_sorted, 'age_final', 'd13c_no_tectonic2')

# d18o
p_d18o_raw,  pow_d18o_raw  = lomb_spec(df_sorted, 'age_final', 'd18o')
p_d18o_36,   pow_d18o_36   = lomb_spec(df_sorted, 'age_final', 'd18o_no_tectonic')
p_d18o_72,   pow_d18o_72   = lomb_spec(df_sorted, 'age_final', 'd18o_no_tectonic2')

# ---- colors ----
greens = cm.get_cmap('Greens')
blues  = cm.get_cmap('Blues')

c_raw_g, c_36_g, c_72_g = greens(0.35), greens(0.65), greens(0.90)
c_raw_b, c_36_b, c_72_b = blues(0.35),  blues(0.65),  blues(0.90)

# ---- plot ----
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True,
                         gridspec_kw={'hspace': 0})

# panel 1: d13c
ax = axes[0]
ax.plot(p_d13c_raw, pow_d13c_raw, color=c_raw_g, lw=1.4, label='Raw')
ax.plot(p_d13c_36,  pow_d13c_36,  color=c_36_g,  lw=1.6, label='36-month filter')
ax.plot(p_d13c_72,  pow_d13c_72,  color=c_72_g,  lw=1.8, label='72-month filter')
ax.axvline(1.0,   color='gray', linestyle='--', alpha=0.6, lw=1.0)
ax.axvline(3.617, color='gray', linestyle='--', alpha=0.6, lw=1.0)
ax.set_ylabel(r'$\delta^{13}$C Spectrum'+'\n'+'Spectral Amplitude')
#ax.set_title(r'$\delta^{13}$C Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0, 0.07)
ax.legend(loc='upper right', frameon=False)
ax.grid(alpha=0.15)

# panel 2: d18o
ax = axes[1]
ax.plot(p_d18o_raw, pow_d18o_raw, color=c_raw_b, lw=1.4, label='Raw')
ax.plot(p_d18o_36,  pow_d18o_36,  color=c_36_b,  lw=1.6, label='36-month filter')
ax.plot(p_d18o_72,  pow_d18o_72,  color=c_72_b,  lw=1.8, label='72-month filter')
ax.axvline(1.0,   color='gray', linestyle='--', alpha=0.6, lw=1.0, label='1 year')
ax.axvline(3.617, color='gray', linestyle='--', alpha=0.6, lw=1.0, label='3.6 years')
ax.set_xlabel('Period (years)')
ax.set_ylabel(r'$\delta^{18}$O Spectrum'+'\n'+'Spectral Amplitude')
#ax.set_title(r'$\delta^{18}$O Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0, 0.2)
ax.legend(loc='upper right', frameon=False)
ax.grid(alpha=0.15)

plt.tight_layout()
#fig.savefig('spectral_d13c_d18o_raw-36m-72m.pdf')
plt.show()
fig

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import correlate

def compute_cross_correlation(x, y, max_lag=None):
    # Drop NaNs jointly
    df = pd.concat([x, y], axis=1).dropna()
    x_clean = df.iloc[:, 0]
    y_clean = df.iloc[:, 1]
    
    # Standardize (important!)
    x_std = (x_clean - x_clean.mean()) / x_clean.std()
    y_std = (y_clean - y_clean.mean()) / y_clean.std()
    
    n = len(x_std)
    
    # Full cross-correlation
    corr = correlate(x_std, y_std, mode='full') / n
    
    lags = np.arange(-n + 1, n)
    
    if max_lag is not None:
        mask = (lags >= -max_lag) & (lags <= max_lag)
        return lags[mask], corr[mask]
    
    return lags, corr


# ---- Compute both cases ----
lags1, corr1 = compute_cross_correlation(
    df_sorted['d13c'],
    df_sorted['d18o'],
    max_lag=100
)

lags2, corr2 = compute_cross_correlation(
    df_sorted['d13c_no_tectonic'],
    df_sorted['d18o_no_tectonic'],
    max_lag=100
)

# ---- Plot ----
plt.figure(figsize=(10, 5))

plt.plot(lags1, corr1, label='Original', linewidth=2)
plt.plot(lags2, corr2, label='No tectonic', linewidth=2)

plt.axvline(0, linestyle='--')
plt.xlabel('Lag')
plt.ylabel('Cross-correlation')
plt.title('Cross-correlation: d13c vs d18o')
plt.legend()

plt.show()

In [ ]:
lags2[np.argmax( corr2)]

In [ ]:
len(corr2)

In [ ]:
df_resampled

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",text=df_sorted['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_final'], y=df_resampled['d13c'],  mode="markers", marker_color="red",text=df_sorted['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_final'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_final'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_final'], y=df_resampled['d13c_ma3'], mode="lines", line_color='black', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_final'], y=df_resampled['d13c_no_tectonic'], mode="lines", line_color='black', name='no tec'), secondary_y=True)


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#F54927', name='Tide Gauge', line_width=1), secondary_y=True)





# Add figure title
fig.update_layout(width=1600, height=900, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
#fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
# Ensure df_resampled is sorted by date
df = df_resampled[['d13c', 'd13c_no_tectonic','d18o','d18o_no_tectonic']].copy()
df = df.sort_index()

# Target dates to align with
#target_dates = pd.to_datetime(enso.loc[:685, 'Date'])
target_dates = pd.date_range('1950-01-31', '2007-01-31', freq='3ME')

# Interpolate to get values at those dates
aligned = df.reindex(df.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])


aligned_enso = enso['ENSO'].reindex(enso['ENSO'].index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_enso.dropna())

aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())

aligned_climate = climate.set_index('datetime')
aligned_climate = aligned_climate.reindex(aligned_climate.index.union(target_dates)).interpolate(method='time').loc[target_dates]

aligned = aligned.join(aligned_climate[['SST','SSS', 'synthetic_d18o']].dropna())

aligned.loc['2005-01-01':'2008-01-01']

In [ ]:
#---------------------------------------FFT-----------------------------------------------------
d13c_fft_freq, d13c_fft = FFT_res(df_resampled['d13c'], 90)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d18o'], 90)

df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window=window_size2, center=True).std()
d13c_stds = df_resampled.dropna()['d13c_std']
d18o_stds = df_resampled.dropna()['d18o_std']
d13c_std_fft_freq, d13c_std_fft = FFT_res(d13c_stds, 90)
d18o_std_fft_freq, d18o_std_fft = FFT_res(d18o_stds, 90)

d13c_aligned_fft_freq, d13c_aligned_fft = FFT_res(aligned['d13c'], 90)
d18o_aligned_fft_freq, d18o_aligned_fft = FFT_res(aligned['d18o'], 90)
#-----------------------------------------------------------------------------------------------

In [ ]:
fig, axes = plt.subplots(3, figsize=(8,6), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_aligned_fft_freq[2:], fft_noramlize(d18o_aligned_fft[2:]),label=r'$\delta^{18}O$')
    if i==1:
        ax.plot(1/d13c_aligned_fft_freq[2:], fft_noramlize(d13c_aligned_fft[2:]), color='green',label=r'$\delta^{13}C$')
    if i==2:
        ax.plot(1/enso_fft_freq[2:], fft_noramlize(enso_fft[2:]), color='gray', label='enso')
    
    ax.legend(loc='upper right')  
    days_freq = 271
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 281
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 746    
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')
    
    ax.set_xlim(300,700)
    #ax.set_xlim(200,3600)
    #ax.set_xscale('log')
    #plt.yticks([0,0.001])

    
axes[0].set_ylim(0,0.003)
axes[1].set_ylim(0,0.04)
axes[2].set_ylim(0,0.6)
ax.set_xlabel('Period [days]')
axes[1].yaxis.tick_right()
axes[1].yaxis.set_label_position("right")
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum ')
axes[1].legend('upper right')
#ax.set_title('Moving Standard Deviation FFT')
fig
#fig.savefig('std_fft.pdf')


In [ ]:
"""
tectonic_climate_spectral.py
─────────────────────────────────────────────────────────────────────────────
Separating tectonic and climate signals in coral δ¹³C and δ¹⁸O.

Produces a 3-row × 2-column figure:
  Row 1 (a/b) : time series with 36m and 72m moving-average overlaid
  Row 2 (c/d) : Lomb–Scargle power in the tectonic band (5–80 yr)
  Row 3 (e/f) : Lomb–Scargle power in the climate band  (0.5–10 yr, log x)

Usage
-----
Edit INPUT_FILE to point to your agemodel_pathsorted.xlsx, then run:
    python tectonic_climate_spectral.py

Outputs
-------
    tectonic_climate_separation.pdf
    tectonic_climate_separation.png
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from astropy.timeseries import LombScargle

# ─── USER SETTINGS ────────────────────────────────────────────────────────────
INPUT_FILE   = 'agemodel_pathsorted.xlsx'   # path to your age-model spreadsheet
OUT_PDF      = 'tectonic_climate_separation.pdf'
OUT_PNG      = 'tectonic_climate_separation.png'

MA_SHORT     = 36    # months — captures ENSO + annual in the MA
MA_LONG      = 72    # months — cleaner tectonic separation (preferred)

FAP_LEVEL    = 0.01  # false-alarm probability threshold (1 %)
# ──────────────────────────────────────────────────────────────────────────────


# ─── HELPERS ──────────────────────────────────────────────────────────────────
def decimal_year_to_datetime(decimal_year):
    """Convert a numpy array of decimal years to pandas DatetimeIndex."""
    year         = decimal_year.astype(int)
    decimal_part = decimal_year - year
    soy  = pd.to_datetime([f'{y}-01-01' for y in year])
    siy  = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - soy).total_seconds()
    secs = decimal_part * siy
    return soy + pd.to_timedelta(secs, unit='s')


def lomb(series_age, series_val, fmin, fmax, npts=5000):
    """
    Compute a Lomb–Scargle periodogram.

    Parameters
    ----------
    series_age, series_val : array-like (must have no NaNs after joint dropna)
    fmin, fmax             : frequency limits in yr⁻¹
    npts                   : number of frequency grid points

    Returns
    -------
    periods : ndarray  (years)
    power   : ndarray
    ls      : LombScargle object (for FAP queries)
    """
    df_tmp = pd.DataFrame({'t': series_age, 'v': series_val}).dropna()
    ls     = LombScargle(df_tmp['t'].values, df_tmp['v'].values)
    freq   = np.linspace(fmin, fmax, npts)
    power  = ls.power(freq)
    return 1.0 / freq, power, ls


def fill_spec(ax, periods, power, color, lw=1.4, alpha_fill=0.15):
    """Plot a spectrum as a filled area + line."""
    ax.fill_between(periods, 0, power, color=color, alpha=alpha_fill)
    ax.plot(periods, power, color=color, lw=lw)


# ─── LOAD & RESAMPLE ──────────────────────────────────────────────────────────
df_raw    = pd.read_excel(INPUT_FILE)
df_sorted = df_raw.sort_values('age_final').reset_index(drop=True)

df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_final'].values)
df_res = (df_sorted[['age_final', 'age_dt', 'd13c', 'd18o']]
          .set_index('age_dt')
          .resample('ME').mean()
          .interpolate())

# ─── BUILD MOVING-AVERAGE AND RESIDUAL COLUMNS ────────────────────────────────
for col in ['d13c', 'd18o']:
    for w in [MA_SHORT, MA_LONG]:
        df_res[f'{col}_ma{w}']    = df_res[col].rolling(w, center=True).mean()
        df_res[f'{col}_notec{w}'] = df_res[col] - df_res[f'{col}_ma{w}']

# ─── COMPUTE ALL SPECTRA ──────────────────────────────────────────────────────
specs = {}
for col in ['d13c', 'd18o',
            f'd13c_ma{MA_SHORT}',  f'd13c_ma{MA_LONG}',
            f'd13c_notec{MA_SHORT}', f'd13c_notec{MA_LONG}',
            f'd18o_ma{MA_SHORT}',
            f'd18o_notec{MA_SHORT}']:
    age_col = df_res['age_final']
    specs[col + '_tect'] = lomb(age_col, df_res[col], 1/80, 1/5,   npts=4000)
    specs[col + '_clim'] = lomb(age_col, df_res[col], 1/10, 1/0.5, npts=6000)

# ─── FALSE ALARM LEVELS ───────────────────────────────────────────────────────
fap = {}
for col in ['d13c', f'd13c_notec{MA_SHORT}', 'd18o', f'd18o_notec{MA_SHORT}']:
    df_tmp = df_res[['age_final', col]].dropna()
    ls_tmp = LombScargle(df_tmp['age_final'].values, df_tmp[col].values)
    fap[col] = ls_tmp.false_alarm_level([FAP_LEVEL])[0]

# ─── PLOT STYLE ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'          : 'sans-serif',
    'font.size'            : 9,
    'axes.linewidth'       : 0.7,
    'xtick.direction'      : 'in',
    'ytick.direction'      : 'in',
    'xtick.top'            : True,
    'ytick.right'          : True,
    'xtick.minor.visible'  : True,
    'ytick.minor.visible'  : True,
})

# Colour palette
G0, G1, G2 = '#c7e3bf', '#4ca64c', '#1a5c1a'   # greens  (d13c)
B0, B1, B2 = '#b8d8f0', '#3a7fc1', '#0d3d6e'   # blues   (d18o)
GREY       = '#888888'
RED        = '#c0392b'
ORANGE     = '#e67e00'

# ─── FIGURE ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 13))
gs  = fig.add_gridspec(3, 2,
                        hspace=0.42, wspace=0.32,
                        height_ratios=[1.6, 1, 1])

ax_ts13 = fig.add_subplot(gs[0, 0])
ax_ts18 = fig.add_subplot(gs[0, 1])
ax_lp13 = fig.add_subplot(gs[1, 0])
ax_lp18 = fig.add_subplot(gs[1, 1])
ax_cl13 = fig.add_subplot(gs[2, 0])
ax_cl18 = fig.add_subplot(gs[2, 1])

# ── Row 0 : Time series ────────────────────────────────────────────────────────
ts_cfg = [
    (ax_ts13, 'd13c', G0, G2, r'$\delta^{13}$C  (‰ VPDB)',   (-3.5,  0.9)),
    (ax_ts18, 'd18o', B0, B2, r'$\delta^{18}$O  (‰ VSMOW)', (24.8, 27.1)),
]
for ax, col, c_raw, c_ma, ylabel, ylim in ts_cfg:
    dr = df_res[['age_final', col]].dropna()
    ax.plot(dr['age_final'], dr[col],
            color=c_raw, lw=0.7, alpha=0.7, label='Monthly mean')

    dm36 = df_res[['age_final', f'{col}_ma{MA_SHORT}']].dropna()
    ax.plot(dm36['age_final'], dm36[f'{col}_ma{MA_SHORT}'],
            color=GREY, lw=1.2, ls='--', alpha=0.75,
            label=f'{MA_SHORT}m MA')

    dm72 = df_res[['age_final', f'{col}_ma{MA_LONG}']].dropna()
    ax.plot(dm72['age_final'], dm72[f'{col}_ma{MA_LONG}'],
            color=c_ma, lw=1.9,
            label=f'{MA_LONG}m MA  (tectonic)')

    ax.set_xlim(1937, 2007)
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel('Year', fontsize=9)
    ax.legend(fontsize=7, frameon=True, framealpha=0.85,
              edgecolor='0.7', loc='lower left')
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))

ax_ts13.set_title(r'$\delta^{13}$C', fontsize=10, pad=4)
ax_ts18.set_title(r'$\delta^{18}$O', fontsize=10, pad=4)

# Annotate secular decline
ax_ts13.annotate(
    '~2.3‰ secular\ndecline',
    xy=(2002, -2.55), xytext=(1972, -3.15),
    fontsize=7.5, color='0.35',
    arrowprops=dict(arrowstyle='->', color='0.4', lw=0.9),
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='0.7', lw=0.6))

# ── Row 1 : Tectonic band (5–80 yr) ───────────────────────────────────────────
ref_lp = [
    (11.0, '#9b59b6', 'Solar ~11yr'),
    (24.0, '#16a085', '~24yr'),
]
lp_cfg = [
    (ax_lp13, 'd13c', f'd13c_ma{MA_SHORT}', f'd13c_ma{MA_LONG}',
     (G0, GREY, G2), 'd13c',   r'$\delta^{13}$C  LS power', (0, 0.40)),
    (ax_lp18, 'd18o', f'd18o_ma{MA_SHORT}', None,
     (B0, GREY, B2), 'd18o',   r'$\delta^{18}$O  LS power', (0, 0.22)),
]
for ax, raw, ma36, ma72, colors, fk, ylabel, ylim in lp_cfg:
    fill_spec(ax, *specs[raw   + '_tect'][:2], colors[0], lw=1.2, alpha_fill=0.12)
    fill_spec(ax, *specs[ma36  + '_tect'][:2], colors[1], lw=1.4, alpha_fill=0.16)
    if ma72:
        fill_spec(ax, *specs[ma72 + '_tect'][:2], colors[2], lw=1.9, alpha_fill=0.22)

    ax.axhline(fap[fk], color='k', ls=':', lw=0.9, alpha=0.6)
    ax.text(78, fap[fk] * 1.04, '1% FAP',
            fontsize=6.5, ha='right', va='bottom', color='0.4')

    for per, col, lbl in ref_lp:
        ax.axvline(per, color=col, ls='--', lw=0.9, alpha=0.7)
        ax.text(per + 0.5, ylim[1] * 0.96, lbl,
                fontsize=6.5, color=col, va='top')

    ax.set_xlim(5, 80)
    ax.set_ylim(*ylim)
    ax.set_xlabel('Period (years)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))

leg_lp = [
    Line2D([0],[0], color=G0,   lw=2,              label='Raw d13c'),
    Line2D([0],[0], color=GREY, lw=1.5,             label=f'{MA_SHORT}m MA'),
    Line2D([0],[0], color=G2,   lw=2,              label=f'{MA_LONG}m MA'),
    Line2D([0],[0], color='k',  lw=1,   ls=':',    label='1% FAP'),
]
ax_lp13.legend(handles=leg_lp, fontsize=7, frameon=True,
               framealpha=0.85, edgecolor='0.7', loc='upper left')

# ── Row 2 : Climate band (0.5–10 yr, log x) ──────────────────────────────────
ref_cl = [
    (1.0,   RED,       'Annual'),
    (3.617, ORANGE,    'ENSO ~3.6yr'),
    (7.0,   '#8e44ad', '~7yr'),
]
cl_cfg = [
    (ax_cl13, 'd13c', f'd13c_notec{MA_SHORT}', f'd13c_notec{MA_LONG}',
     (G0, G1, G2), f'd13c_notec{MA_SHORT}', r'$\delta^{13}$C  LS power', (0, 0.080)),
    (ax_cl18, 'd18o', f'd18o_notec{MA_SHORT}', None,
     (B0, B1, None), f'd18o_notec{MA_SHORT}', r'$\delta^{18}$O  LS power', (0, 0.220)),
]
for ax, raw, notec36, notec72, colors, fk, ylabel, ylim in cl_cfg:
    fill_spec(ax, *specs[raw     + '_clim'][:2], colors[0], lw=1.2, alpha_fill=0.12)
    fill_spec(ax, *specs[notec36 + '_clim'][:2], colors[1], lw=1.6, alpha_fill=0.20)
    if notec72:
        fill_spec(ax, *specs[notec72 + '_clim'][:2], colors[2], lw=1.9, alpha_fill=0.22)

    ax.axhline(fap[fk], color='k', ls=':', lw=0.9, alpha=0.6)
    ax.text(9.3, fap[fk] * 1.04, '1% FAP',
            fontsize=6.5, ha='right', va='bottom', color='0.4')

    for per, col, lbl in ref_cl:
        ax.axvline(per, color=col, ls='--', lw=0.9, alpha=0.7)
        ax.text(per * 1.06, ylim[1] * 0.96, lbl,
                fontsize=6.5, color=col, va='top')

    ax.set_xlim(0.5, 10)
    ax.set_ylim(*ylim)
    ax.set_xscale('log')
    ax.set_xlabel('Period (years) — log scale', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f'{x:.0f}' if x >= 1 else f'{x:.1f}'))

leg_cl = [
    Line2D([0],[0], color=G0, lw=2,           label='Raw'),
    Line2D([0],[0], color=G1, lw=2,           label=f'−{MA_SHORT}m MA'),
    Line2D([0],[0], color=G2, lw=2,           label=f'−{MA_LONG}m MA'),
    Line2D([0],[0], color='k', lw=1, ls=':',  label='1% FAP'),
]
ax_cl13.legend(handles=leg_cl, fontsize=7, frameon=True,
               framealpha=0.85, edgecolor='0.7', loc='upper right')

# ── Panel labels and row annotations ──────────────────────────────────────────
for ax, lbl in zip(
        [ax_ts13, ax_ts18, ax_lp13, ax_lp18, ax_cl13, ax_cl18],
        ['(a)',   '(b)',   '(c)',   '(d)',   '(e)',   '(f)']):
    ax.text(0.012, 0.97, lbl, transform=ax.transAxes,
            fontsize=8.5, va='top', fontweight='bold')

for ax, lbl in [(ax_lp13, 'Tectonic band\n(5–80 yr)'),
                (ax_cl13, 'Climate band\n(0.5–10 yr)')]:
    ax.annotate(lbl, xy=(1.025, 0.5), xycoords='axes fraction',
                fontsize=7.5, ha='left', va='center', color='0.4')

fig.suptitle(
    r'Separating tectonic and climate signals — $\delta^{13}$C and $\delta^{18}$O',
    fontsize=11, y=0.998)

fig
# ── Save ──────────────────────────────────────────────────────────────────────
#plt.savefig(OUT_PDF, bbox_inches='tight', dpi=200)
#plt.savefig(OUT_PNG, bbox_inches='tight', dpi=180)
#print(f'Saved  {OUT_PDF}  and  {OUT_PNG}')

In [ ]:
import numpy
from matplotlib import pyplot

import pycwt as wavelet
from pycwt.helpers import find

In [ ]:
df_resampled

In [ ]:
data = df_resampled['d13c_no_tectonic'].dropna()
dat = data.values
par_name = 'd13c'
title = "d13C without tectonic signal"
label = "d13C"
units = ""
t0 = datetime_to_decimal_year(data.index[0])
dt = data.index.diff().mean().total_seconds()/((365*24+6)*60*60)

N = dat.size
#t = numpy.arange(0, N) * dt + t0
t = data.index.year+(1+data.index.month)/12-1/365
locals()['t_'+par_name] = t


p = numpy.polyfit(t - t0, dat, 1)
dat_notrend = dat - numpy.polyval(p, t - t0)
std = dat_notrend.std()  # Standard deviation
var = std**2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset
locals()['dat_norm_'+par_name] = dat_norm

mother = wavelet.Morlet(6)
s0 =  2*dt  # Starting scale, in this case 2 * dt years = 2 months
dj = 1 / 24  # Twelve sub-octaves per octaves
J = 10 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (numpy.abs(wave)) ** 2
fft_power = numpy.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]

In [ ]:
signif, fft_theor = wavelet.significance(
    1.0, dt, scales, 0, alpha, significance_level=0.95, wavelet=mother
)
sig95 = numpy.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95


glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(
    var, dt, scales, 1, alpha, significance_level=0.95, dof=dof, wavelet=mother
)

sel=np.array([1,2])
Cdelta = mother.cdelta
scale_avg = (scales * numpy.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(
    var,
    dt,
    scales,
    2,
    alpha,
    significance_level=0.95,
    dof=[scales[sel[0]], scales[sel[-1]]],
    wavelet=mother,
)

In [ ]:
# Prepare the figure
pyplot.close("all")
pyplot.ioff()
figprops = dict(figsize=(11, 8), dpi=72)
fig = pyplot.figure(**figprops)

# First sub-plot, the original time series anomaly and inverse wavelet
# transform.
ax = pyplot.axes([0.1, 0.75, 0.65, 0.2])
ax.plot(t, iwave, "-", linewidth=1, color=[0.5, 0.5, 0.5])
ax.plot(t, dat, "k", linewidth=1.5)
ax.set_title("a) {}".format(title))
ax.set_ylabel(r"{} [{}]".format(label, units))

# Second sub-plot, the normalized wavelet power spectrum and significance
# level contour lines and cone of influece hatched area. Note that period
# scale is logarithmic.
bx = pyplot.axes([0.1, 0.37, 0.65, 0.28], sharex=ax)
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
bx.contourf(t, numpy.log2(period), numpy.log2(power), numpy.log2(levels), extend="both", cmap=pyplot.cm.viridis)
extent = [t.min(), t.max(), 0, max(period)]
bx.contour(t, numpy.log2(period), sig95, [-99, 1], colors="k", linewidths=2, extent=extent)
bx.fill(numpy.concatenate([t, t[-1:] + dt, t[-1:] + dt, t[:1] - dt, t[:1] - dt]),
    numpy.concatenate([numpy.log2(coi), [1e-9],  numpy.log2(period[-1:]), numpy.log2(period[-1:]), [1e-9],]),"k",  alpha=0.3, hatch="x")
bx.set_title("b) {} Wavelet Power Spectrum ({})".format(label, mother.name))
bx.set_ylabel("Period (years)")
bx.hlines(numpy.log2(1), t[0], t[-1], colors='orange', linestyles='dashed')

#
Yticks = 2 ** numpy.arange(numpy.ceil(numpy.log2(period.min())), numpy.ceil(numpy.log2(period.max())))
bx.set_yticks(numpy.log2(Yticks))
bx.set_yticklabels(Yticks)

# Third sub-plot, the global wavelet and Fourier power spectra and theoretical
# noise spectra. Note that period scale is logarithmic.
cx = pyplot.axes([0.77, 0.37, 0.2, 0.28], sharey=bx)
cx.plot(glbl_signif, numpy.log2(period), "k--")
cx.plot(var * fft_theor, numpy.log2(period), "--", color="#cccccc")
cx.plot( var * fft_power, numpy.log2(1.0 / fftfreqs), "-", color="#cccccc", linewidth=1.0)
cx.plot(var * glbl_power, numpy.log2(period), "k-", linewidth=1.5)
cx.set_title("c) Global Wavelet Spectrum")
cx.set_xlabel(r"Power [({})^2]".format(units))
cx.set_xlim([0, glbl_power.max() + var])
cx.set_ylim(numpy.log2([period.min(), period.max()]))
cx.set_yticks(numpy.log2(Yticks))
cx.set_yticklabels(Yticks)
pyplot.setp(cx.get_yticklabels(), visible=False)

# Fourth sub-plot, the scale averaged wavelet spectrum.
dx = pyplot.axes([0.1, 0.07, 0.65, 0.2], sharex=ax)
dx.axhline(scale_avg_signif, color="k", linestyle="--", linewidth=1.0)
dx.plot(t, scale_avg, "k-", linewidth=1.5)
dx.set_title("d) {}--{} year scale-averaged power".format(sel[0], sel[1]))
dx.set_xlabel("Time (year)")
dx.set_ylabel(r"Average variance [{}]".format(units))
ax.set_xlim([t.min(), t.max()])

pyplot.show()
fig
#fig.savefig('wavelet'+label+'.pdf')

In [ ]:
df_sorted['d13c']

In [ ]:
# ── Sliding-Window Lomb-Scargle (SWLS) spectrogram ────────────────────────────
# Works directly on df_sorted (raw irregular data) — no resampling needed.
# At each time grid point, a Gaussian-weighted LS periodogram is computed
# on the raw samples within the window. Weights decay as exp(-0.5*(Δt/σ)²),
# passed to astropy as dy = 1/sqrt(w) so high-weight points constrain the fit.
#
# Tectonic MA: time-based ±1.5yr average on raw data (same fix as wavelet cell).

from astropy.timeseries import LombScargle
import numpy as np

# ── Parameters ────────────────────────────────────────────────────────────────
SIGMA    = 2.0    # Gaussian window width (years) — trades time vs freq resolution
HALF_WIN = 1.5    # ±1.5yr tectonic MA window (= 36 months)
MIN_PTS  = 5      # min raw points in window for a valid LS estimate
COV_THRESH = 20   # points below this are flagged as low-coverage

# Period grid: 0.5yr to 15yr, log-spaced
periods = np.logspace(np.log10(0.5), np.log10(15), 120)
freqs   = 1.0 / periods

# Time output grid: 3-month steps
t_grid = np.arange(1940.0, 2007.0, 0.25)

# ── Raw data ──────────────────────────────────────────────────────────────────
t_raw = df_sorted['age_final'].values
y13c  = df_sorted['d13c'].values
y18o  = df_sorted['d18o'].values

# ── Tectonic MA on raw data, interpolated back to raw times ───────────────────
def time_based_ma_at_raw(t_raw, y_raw, t_grid, half_win=1.5, min_pts=3):
    """Time-windowed MA evaluated at t_grid, then interpolated back to t_raw."""
    ma_grid = np.full(len(t_grid), np.nan)
    for i, tc in enumerate(t_grid):
        m = (t_raw >= tc - half_win) & (t_raw <= tc + half_win)
        if m.sum() >= min_pts:
            ma_grid[i] = np.nanmean(y_raw[m])
    valid = ~np.isnan(ma_grid)
    return np.interp(t_raw, t_grid[valid], ma_grid[valid]), ma_grid

ma13c_at_raw, ma13c_grid = time_based_ma_at_raw(t_raw, y13c, t_grid)
ma18o_at_raw, ma18o_grid = time_based_ma_at_raw(t_raw, y18o, t_grid)

notec13c = y13c - ma13c_at_raw
notec18o = y18o - ma18o_at_raw

# ── SWLS core ─────────────────────────────────────────────────────────────────
def swls(t_raw, y_raw, t_grid, freqs, sigma, min_pts):
    """Sliding-window Lomb-Scargle spectrogram on raw irregular data."""
    power_map = np.full((len(freqs), len(t_grid)), np.nan)
    npts_map  = np.zeros(len(t_grid), dtype=int)
    for i, tc in enumerate(t_grid):
        w    = np.exp(-0.5 * ((t_raw - tc) / sigma)**2)
        mask = w > 0.01          # ignore points beyond ~3σ
        npts_map[i] = mask.sum()
        if mask.sum() < min_pts:
            continue
        ls = LombScargle(t_raw[mask], y_raw[mask], dy=1.0/np.sqrt(w[mask]))
        power_map[:, i] = ls.power(freqs)
    return power_map, npts_map

power_13c, npts = swls(t_raw, notec13c, t_grid, freqs, SIGMA, MIN_PTS)
power_18o, _    = swls(t_raw, notec18o, t_grid, freqs, SIGMA, MIN_PTS)

# ── Global LS for the sidebar spectrum ────────────────────────────────────────
ls_13c   = LombScargle(t_raw, notec13c)
ls_18o   = LombScargle(t_raw, notec18o)
glbl_13c = ls_13c.power(freqs)
glbl_18o = ls_18o.power(freqs)
fap_13c  = ls_13c.false_alarm_level([0.01])[0]
fap_18o  = ls_18o.false_alarm_level([0.01])[0]
cov_mask = npts < COV_THRESH

In [ ]:
"""
swls_plot.py
─────────────────────────────────────────────────────────────────────────────
Plotting code for the Sliding-Window Lomb-Scargle (SWLS) spectrogram.

Expects the following variables to already exist in the namespace
(produced by the SWLS computation cell):

    t_raw       — raw sample times (decimal years)
    y13c        — raw δ¹³C values
    y18o        — raw δ¹⁸O values
    ma13c_grid  — tectonic MA evaluated on t_grid  (shape: N_t)
    ma18o_grid  — same for δ¹⁸O
    t_grid      — uniform output time grid (decimal years)
    periods     — period array (years, log-spaced)
    power_13c   — SWLS power map (N_periods × N_times)
    power_18o   — same for δ¹⁸O
    npts        — number of raw pts per window (N_times)
    glbl_13c    — global LS power for δ¹³C  (N_periods)
    glbl_18o    — global LS power for δ¹⁸O  (N_periods)
    fap_13c     — 1% FAP level for global δ¹³C LS  (scalar)
    fap_18o     — 1% FAP level for global δ¹⁸O LS  (scalar)
    cov_mask    — boolean mask: True where coverage < COV_THRESH  (N_times)

Usage in notebook
-----------------
Run the SWLS computation cell first, then paste this cell below it.
Or run as a standalone script after setting the variables above.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# ─── PLOT SETTINGS ────────────────────────────────────────────────────────────
OUT_PDF = 'swls_d13c_d18o.pdf'
OUT_PNG = 'swls_d13c_d18o.png'
SIGMA   = 2.0   # must match the value used in the computation cell (for footer)

# Reference periods annotated on the scalogram
REF_PERIODS = [
    (1.000, '#e74c3c', 'Annual'),
    (3.617, '#e67e00', 'ENSO'),
    (11.0,  '#8e44ad', '~11yr'),
]

G1 = '#4ca64c'   # green  (δ¹³C)
B1 = '#3a7fc1'   # blue   (δ¹⁸O)
# ──────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    'font.family'         : 'sans-serif',
    'font.size'           : 9,
    'axes.linewidth'      : 0.7,
    'xtick.direction'     : 'in',
    'ytick.direction'     : 'in',
    'xtick.top'           : True,
    'ytick.right'         : True,
    'xtick.minor.visible' : True,
})

# ── Shared y-axis quantities (log₂ period) ────────────────────────────────────
Yticks = 2 ** np.arange(
    np.ceil(np.log2(periods.min())),
    np.ceil(np.log2(periods.max())))
Yticks = Yticks[(Yticks >= 0.5) & (Yticks <= 15)]
y_min  = float(np.log2(periods.min()))
y_max  = float(np.log2(periods.max()))


# ─── PANEL-DRAWING FUNCTION ───────────────────────────────────────────────────
def make_swls_panel(fig, pos_ts, pos_wvt, pos_glbl,
                    t_raw, y_raw, ma_grid, t_grid,
                    power, cov_mask, glbl, fap, color, label):
    """
    Draw one column of SWLS panels (time series / spectrogram / global LS).

    Parameters
    ----------
    fig           : matplotlib Figure
    pos_ts        : [left, bottom, width, height] for time-series panel
    pos_wvt       : [left, bottom, width, height] for spectrogram panel
    pos_glbl      : [left, bottom, width, height] for global LS panel
    t_raw         : raw sample times (decimal years)
    y_raw         : raw signal values
    ma_grid       : tectonic MA on t_grid  (NaN where insufficient data)
    t_grid        : uniform output time grid
    power         : SWLS power map (N_periods × N_times)
    cov_mask      : boolean mask, True = low coverage
    glbl          : global LS power (N_periods)
    fap           : global 1% FAP level (scalar)
    color         : line / scatter colour
    label         : y-axis label string

    Returns
    -------
    ax_ts, ax_sc, ax_gl, cf  — axes handles and contourf QuadContourSet
    """

    # ── (i) time series ───────────────────────────────────────────────────────
    ax_ts = fig.add_axes(pos_ts)
    ax_ts.scatter(t_raw, y_raw, color=color, s=4, alpha=0.4, zorder=2,
                  label='Raw samples')
    vm = ~np.isnan(ma_grid)
    ax_ts.plot(t_grid[vm], ma_grid[vm], 'k', lw=1.8,
               label='Tectonic MA (±1.5yr)', zorder=3)
    ax_ts.set_xlim(t_grid[0], t_grid[-1])
    ax_ts.set_ylabel(label, fontsize=8)
    ax_ts.xaxis.set_minor_locator(ticker.MultipleLocator(5))
    #ax_ts.set_xticklabels([])
    ax_ts.legend(fontsize=7, frameon=False, loc='lower left')

    # ── (ii) spectrogram ──────────────────────────────────────────────────────
    ax_sc = fig.add_axes(pos_wvt, sharex=ax_ts)

    # Replace zeros/negatives with NaN for log scaling
    pw = np.where(power > 0, power, np.nan)

    # Percentile-based colour scale (robust against outliers)
    vmin   = float(np.nanpercentile(pw, 2))
    vmax   = float(np.nanpercentile(pw, 98))
    levels = np.logspace(np.log10(max(vmin, 1e-6)), np.log10(vmax), 40)

    cf = ax_sc.contourf(
        t_grid, np.log2(periods), np.log2(pw),
        np.log2(levels), extend='both', cmap='viridis')

    # White contour where local power exceeds the global 1% FAP level
    # (conservative — not a true local significance test, but informative)
    ax_sc.contour(
        t_grid, np.log2(periods), power,
        [fap], colors='white', linewidths=1.5)

    # Shade low-coverage time steps
    for i in np.where(cov_mask)[0]:
        ax_sc.axvspan(t_grid[i] - 0.13, t_grid[i] + 0.13,
                      color='k', alpha=0.25, lw=0)

    # Reference period lines
    for per, col, _ in REF_PERIODS:
        ax_sc.axhline(np.log2(per), color=col, ls='--', lw=1.0, alpha=0.85)

    ax_sc.set_ylim(y_min, y_max)
    ax_sc.set_yticks(np.log2(Yticks))
    ax_sc.set_yticklabels(
        [f'{y:.0f}' if y >= 1 else f'{y:.1f}' for y in Yticks])
    ax_sc.set_ylabel('Period (years)', fontsize=8)
    ax_sc.set_xlabel('Year', fontsize=9)
    ax_sc.xaxis.set_minor_locator(ticker.MultipleLocator(5))

    # ── (iii) global LS spectrum ──────────────────────────────────────────────
    ax_gl = fig.add_axes(pos_glbl, sharey=ax_sc)
    ax_gl.plot(glbl, np.log2(periods), color=color, lw=1.8, label='Global LS')
    ax_gl.axvline(fap, color='k', ls='--', lw=1.0, label='1% FAP')
    ax_gl.set_ylim(y_min, y_max)
    ax_gl.set_yticks(np.log2(Yticks))
    plt.setp(ax_gl.get_yticklabels(), visible=False)
    ax_gl.set_xlabel('LS power', fontsize=8)
    ax_gl.set_title('Global', fontsize=8, pad=2)
    ax_gl.legend(fontsize=6.5, frameon=False)

    for per, col, _ in REF_PERIODS:
        ax_gl.axhline(np.log2(per), color=col, ls='--', lw=1.0, alpha=0.85)

    return ax_ts, ax_sc, ax_gl, cf


# ─── BUILD FIGURE ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 11))

# Absolute axis positions [left, bottom, width, height]
L    = 0.07
W    = 0.38    # spectrogram width
GW   = 0.10   # global-spectrum width
GAP  = 0.12   # gap between columns
R2   = L + W + GW + GAP   # left edge of right column

TS_H = 0.13   # time-series panel height
SC_H = 0.36   # spectrogram height
BOT  = 0.08   # bottom margin
SC_B = BOT
TS_B = BOT + SC_H + 0.04

# δ¹³C (left column)
ax_ts13, ax_sc13, ax_gl13, cf13 = make_swls_panel(
    fig,
    pos_ts   = [L,        TS_B, W,  TS_H],
    pos_wvt  = [L,        SC_B, W,  SC_H],
    pos_glbl = [L+W+0.01, SC_B, GW, SC_H],
    t_raw=t_raw, y_raw=y13c,
    ma_grid=ma13c_grid, t_grid=t_grid,
    power=power_13c, cov_mask=cov_mask,
    glbl=glbl_13c, fap=fap_13c,
    color=G1, label=r'$\delta^{13}$C  (‰)')

# δ¹⁸O (right column)
ax_ts18, ax_sc18, ax_gl18, cf18 = make_swls_panel(
    fig,
    pos_ts   = [R2,        TS_B, W,  TS_H],
    pos_wvt  = [R2,        SC_B, W,  SC_H],
    pos_glbl = [R2+W+0.01, SC_B, GW, SC_H],
    t_raw=t_raw, y_raw=y18o,
    ma_grid=ma18o_grid, t_grid=t_grid,
    power=power_18o, cov_mask=cov_mask,
    glbl=glbl_18o, fap=fap_18o,
    color=B1, label=r'$\delta^{18}$O  (‰)')

# ── Shared colourbar ──────────────────────────────────────────────────────────
cbar_ax = fig.add_axes(
    [L, TS_B + TS_H + 0.03, W * 2 + GW * 2 + GAP, 0.012])
cb = plt.colorbar(cf13, cax=cbar_ax, orientation='horizontal')
cb.set_label('log₂ (LS power in sliding window)', fontsize=8)
cb.ax.tick_params(labelsize=7)

# ── Column titles ─────────────────────────────────────────────────────────────
ax_ts13.set_title(
    r'$\delta^{13}$C  —  raw irregular data, no resampling',
    fontsize=9, pad=4)
ax_ts18.set_title(
    r'$\delta^{18}$O  —  raw irregular data, no resampling',
    fontsize=9, pad=4)

# ── Legend for reference lines + annotations ──────────────────────────────────
legend_els = [
    Line2D([0], [0], color=col, lw=1.5, ls='--', label=lbl)
    for _, col, lbl in REF_PERIODS
]
legend_els += [
    Line2D([0], [0], color='white', lw=1.5, label='Global 1% FAP contour'),
    plt.fill([], [], color='k', alpha=0.25,
             label='Low coverage (<20 pts)')[0],
]
ax_sc13.legend(
    handles=legend_els, fontsize=6.5,
    frameon=True, framealpha=0.75, edgecolor='0.6',
    facecolor='0.15', loc='upper left')

# ── Panel labels ──────────────────────────────────────────────────────────────
for ax, lbl, white in [
    (ax_ts13, '(a)', False), (ax_ts18, '(b)', False),
    (ax_sc13, '(c)', True),  (ax_sc18, '(d)', True),
]:
    ax.text(0.01, 0.97, lbl, transform=ax.transAxes,
            fontsize=8.5, va='top', fontweight='bold',
            color='white' if white else 'black')

# ── Footer note ───────────────────────────────────────────────────────────────
fig.text(
    0.5, 0.005,
    f'Sliding-window Lomb-Scargle (SWLS, σ={SIGMA}yr Gaussian): '
    'weighted LS on raw irregular samples at each time grid point. '
    'No resampling. Tectonic removed via ±1.5yr time-based MA on raw data. '
    'White contour = global 1% FAP level (conservative local indicator).',
    ha='center', fontsize=7, color='0.4', style='italic')

# ── Save ──────────────────────────────────────────────────────────────────────
plt.savefig(OUT_PDF, bbox_inches='tight', dpi=200)
plt.savefig(OUT_PNG, bbox_inches='tight', dpi=180)
print(f'Saved  {OUT_PDF}  and  {OUT_PNG}')
fig

In [ ]:
"""
tectonic_climate_spectral.py
─────────────────────────────────────────────────────────────────────────────
Separating tectonic and climate signals in coral δ¹³C and δ¹⁸O.

Produces a 3-row × 2-column figure:
  Row 1 (a/b) : time series with 36m and 72m moving-average overlaid
  Row 2 (c/d) : Lomb–Scargle power in the tectonic band (5–80 yr)
  Row 3 (e/f) : Lomb–Scargle power in the climate band  (0.5–10 yr, log x)

Usage
-----
Edit INPUT_FILE to point to your agemodel_pathsorted.xlsx, then run:
    python tectonic_climate_spectral.py

Outputs
-------
    tectonic_climate_separation.pdf
    tectonic_climate_separation.png
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from astropy.timeseries import LombScargle

# ─── USER SETTINGS ────────────────────────────────────────────────────────────
INPUT_FILE   = 'agemodel_pathsorted.xlsx'   # path to your age-model spreadsheet
OUT_PDF      = 'tectonic_climate_separation.pdf'
OUT_PNG      = 'tectonic_climate_separation.png'

MA_SHORT     = 36    # months — captures ENSO + annual in the MA
MA_LONG      = 72    # months — cleaner tectonic separation (preferred)

FAP_LEVEL    = 0.01  # false-alarm probability threshold (1 %)
# ──────────────────────────────────────────────────────────────────────────────


# ─── HELPERS ──────────────────────────────────────────────────────────────────
def decimal_year_to_datetime(decimal_year):
    """Convert a numpy array of decimal years to pandas DatetimeIndex."""
    year         = decimal_year.astype(int)
    decimal_part = decimal_year - year
    soy  = pd.to_datetime([f'{y}-01-01' for y in year])
    siy  = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - soy).total_seconds()
    secs = decimal_part * siy
    return soy + pd.to_timedelta(secs, unit='s')


def lomb(series_age, series_val, fmin, fmax, npts=5000):
    """
    Compute a Lomb–Scargle periodogram.

    Parameters
    ----------
    series_age, series_val : array-like (must have no NaNs after joint dropna)
    fmin, fmax             : frequency limits in yr⁻¹
    npts                   : number of frequency grid points

    Returns
    -------
    periods : ndarray  (years)
    power   : ndarray
    ls      : LombScargle object (for FAP queries)
    """
    df_tmp = pd.DataFrame({'t': series_age, 'v': series_val}).dropna()
    ls     = LombScargle(df_tmp['t'].values, df_tmp['v'].values)
    freq   = np.linspace(fmin, fmax, npts)
    power  = ls.power(freq)
    return 1.0 / freq, power, ls


def fill_spec(ax, periods, power, color, lw=1.4, alpha_fill=0.15):
    """Plot a spectrum as a filled area + line."""
    ax.fill_between(periods, 0, power, color=color, alpha=alpha_fill)
    ax.plot(periods, power, color=color, lw=lw)


# ─── LOAD RAW DATA ────────────────────────────────────────────────────────────
df_raw    = pd.read_excel(INPUT_FILE)
df_sorted = df_raw.sort_values('age_final').reset_index(drop=True)

# Monthly resample is kept ONLY for the time-series panels (smooth MA curves).
# All Lomb-Scargle periodograms are computed on df_sorted (raw irregular data)
# to avoid inflating N with linearly-interpolated monthly values, which would
# artificially lower the FAP threshold.
df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_final'].values)
df_res = (df_sorted[['age_final', 'age_dt', 'd13c', 'd18o']]
          .set_index('age_dt')
          .resample('ME').mean()
          .interpolate())

# ─── MOVING-AVERAGE COLUMNS ───────────────────────────────────────────────────
# On monthly grid — used only for time-series plot overlays
for col in ['d13c', 'd18o']:
    for w in [MA_SHORT, MA_LONG]:
        df_res[f'{col}_ma{w}'] = df_res[col].rolling(w, center=True).mean()

# On raw irregular data — used for all LS periodograms.
# Time-based MA: for each raw sample, average all raw points within ±half-window.
# This keeps only real measurements in the average (no interpolated values).
_half = MA_SHORT / 12 / 2    # MA_SHORT months → years, then half-window
t_raw = df_sorted['age_final'].values
for col in ['d13c', 'd18o']:
    y_raw = df_sorted[col].values
    ma_vals = np.array([
        np.nanmean(y_raw[np.abs(t_raw - tc) <= _half])
        for tc in t_raw
    ])
    df_sorted[f'{col}_ma{MA_SHORT}']    = ma_vals
    df_sorted[f'{col}_notec{MA_SHORT}'] = y_raw - ma_vals

_half_long = MA_LONG / 12 / 2
for col in ['d13c', 'd18o']:
    y_raw = df_sorted[col].values
    ma_vals = np.array([
        np.nanmean(y_raw[np.abs(t_raw - tc) <= _half_long])
        for tc in t_raw
    ])
    df_sorted[f'{col}_ma{MA_LONG}']    = ma_vals
    df_sorted[f'{col}_notec{MA_LONG}'] = y_raw - ma_vals

# ─── COMPUTE ALL SPECTRA (on raw df_sorted) ───────────────────────────────────
specs = {}
for col in ['d13c', 'd18o',
            f'd13c_ma{MA_SHORT}',    f'd13c_ma{MA_LONG}',
            f'd13c_notec{MA_SHORT}', f'd13c_notec{MA_LONG}',
            f'd18o_ma{MA_SHORT}',
            f'd18o_notec{MA_SHORT}']:
    age_col = df_sorted['age_final']
    specs[col + '_tect'] = lomb(age_col, df_sorted[col], 1/80, 1/5,   npts=4000)
    specs[col + '_clim'] = lomb(age_col, df_sorted[col], 1/10, 1/0.5, npts=6000)

# ─── FALSE ALARM LEVELS (on raw df_sorted) ────────────────────────────────────
fap = {}
for col in ['d13c', f'd13c_notec{MA_SHORT}', 'd18o', f'd18o_notec{MA_SHORT}']:
    df_tmp = df_sorted[['age_final', col]].dropna()
    ls_tmp = LombScargle(df_tmp['age_final'].values, df_tmp[col].values)
    fap[col] = ls_tmp.false_alarm_level([FAP_LEVEL])[0]

# ─── PLOT STYLE ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'          : 'sans-serif',
    'font.size'            : 9,
    'axes.linewidth'       : 0.7,
    'xtick.direction'      : 'in',
    'ytick.direction'      : 'in',
    'xtick.top'            : True,
    'ytick.right'          : True,
    'xtick.minor.visible'  : True,
    'ytick.minor.visible'  : True,
})

# Colour palette
G0, G1, G2 = '#c7e3bf', '#4ca64c', '#1a5c1a'   # greens  (d13c)
B0, B1, B2 = '#b8d8f0', '#3a7fc1', '#0d3d6e'   # blues   (d18o)
GREY       = '#888888'
RED        = '#c0392b'
ORANGE     = '#e67e00'

# ─── FIGURE ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 11))
gs  = fig.add_gridspec(3, 2,
                        hspace=0.42, wspace=0.32,
                        height_ratios=[1.6, 1, 1])

ax_ts13 = fig.add_subplot(gs[0, 0])
ax_ts18 = fig.add_subplot(gs[0, 1])
ax_lp13 = fig.add_subplot(gs[1, 0])
ax_lp18 = fig.add_subplot(gs[1, 1])
ax_cl13 = fig.add_subplot(gs[2, 0])
ax_cl18 = fig.add_subplot(gs[2, 1])

# ── Row 0 : Time series ────────────────────────────────────────────────────────
ts_cfg = [
    (ax_ts13, 'd13c', G0, G2, r'$\delta^{13}$C  (‰ VPDB)',   (-3.5,  0.9)),
    (ax_ts18, 'd18o', B0, B2, r'$\delta^{18}$O  (‰ VSMOW)', (24.8, 27.1)),
]
for ax, col, c_raw, c_ma, ylabel, ylim in ts_cfg:
    dr = df_res[['age_final', col]].dropna()
    ax.plot(dr['age_final'], dr[col],
            color=c_raw, lw=0.7, alpha=0.7, label='Monthly mean')

    dm36 = df_res[['age_final', f'{col}_ma{MA_SHORT}']].dropna()
    ax.plot(dm36['age_final'], dm36[f'{col}_ma{MA_SHORT}'],
            color=GREY, lw=1.2, ls='--', alpha=0.75,
            label=f'{MA_SHORT}m MA')

    dm72 = df_res[['age_final', f'{col}_ma{MA_LONG}']].dropna()
    ax.plot(dm72['age_final'], dm72[f'{col}_ma{MA_LONG}'],
            color=c_ma, lw=1.9,
            label=f'{MA_LONG}m MA  (tectonic)')

    ax.set_xlim(1937, 2007)
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel('Year', fontsize=9)
    ax.legend(fontsize=7, frameon=True, framealpha=0.85,
              edgecolor='0.7', loc='lower left')
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))

ax_ts13.set_title(r'$\delta^{13}$C', fontsize=10, pad=4)
ax_ts18.set_title(r'$\delta^{18}$O', fontsize=10, pad=4)

# Annotate secular decline
ax_ts13.annotate(
    '~2.3‰ secular\ndecline',
    xy=(2002, -2.55), xytext=(1972, -3.15),
    fontsize=7.5, color='0.35',
    arrowprops=dict(arrowstyle='->', color='0.4', lw=0.9),
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='0.7', lw=0.6))

# ── Row 1 : Tectonic band (5–80 yr) ───────────────────────────────────────────
ref_lp = [
    (11.0, '#9b59b6', 'Solar ~11yr'),
    (24.0, '#16a085', '~24yr'),
]
lp_cfg = [
    (ax_lp13, 'd13c', f'd13c_ma{MA_SHORT}', f'd13c_ma{MA_LONG}',
     (G0, GREY, G2), 'd13c',   r'$\delta^{13}$C  LS power', (0, 0.40)),
    (ax_lp18, 'd18o', f'd18o_ma{MA_SHORT}', None,
     (B0, GREY, B2), 'd18o',   r'$\delta^{18}$O  LS power', (0, 0.22)),
]
for ax, raw, ma36, ma72, colors, fk, ylabel, ylim in lp_cfg:
    fill_spec(ax, *specs[raw   + '_tect'][:2], colors[0], lw=1.2, alpha_fill=0.12)
    fill_spec(ax, *specs[ma36  + '_tect'][:2], colors[1], lw=1.4, alpha_fill=0.16)
    if ma72:
        fill_spec(ax, *specs[ma72 + '_tect'][:2], colors[2], lw=1.9, alpha_fill=0.22)

    ax.axhline(fap[fk], color='k', ls=':', lw=0.9, alpha=0.6)
    ax.text(78, fap[fk] * 1.04, '1% FAP',
            fontsize=6.5, ha='right', va='bottom', color='0.4')

    for per, col, lbl in ref_lp:
        ax.axvline(per, color=col, ls='--', lw=0.9, alpha=0.7)
        ax.text(per + 0.5, ylim[1] * 0.96, lbl,
                fontsize=6.5, color=col, va='top')

    ax.set_xlim(5, 80)
    ax.set_ylim(*ylim)
    ax.set_xlabel('Period (years)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))

leg_lp = [
    Line2D([0],[0], color=G0,   lw=2,              label='Raw d13c'),
    Line2D([0],[0], color=GREY, lw=1.5,             label=f'{MA_SHORT}m MA'),
    Line2D([0],[0], color=G2,   lw=2,              label=f'{MA_LONG}m MA'),
    Line2D([0],[0], color='k',  lw=1,   ls=':',    label='1% FAP'),
]
ax_lp13.legend(handles=leg_lp, fontsize=7, frameon=True,
               framealpha=0.85, edgecolor='0.7', loc='upper left')

# ── Row 2 : Climate band (0.5–10 yr, log x) ──────────────────────────────────
ref_cl = [
    (1.0,   RED,       'Annual'),
    (3.617, ORANGE,    'ENSO ~3.6yr'),
    (7.0,   '#8e44ad', '~7yr'),
]
cl_cfg = [
    (ax_cl13, 'd13c', f'd13c_notec{MA_SHORT}', f'd13c_notec{MA_LONG}',
     (G0, G1, G2), f'd13c_notec{MA_SHORT}', r'$\delta^{13}$C  LS power', (0, 0.080)),
    (ax_cl18, 'd18o', f'd18o_notec{MA_SHORT}', None,
     (B0, B1, None), f'd18o_notec{MA_SHORT}', r'$\delta^{18}$O  LS power', (0, 0.220)),
]
for ax, raw, notec36, notec72, colors, fk, ylabel, ylim in cl_cfg:
    fill_spec(ax, *specs[raw     + '_clim'][:2], colors[0], lw=1.2, alpha_fill=0.12)
    fill_spec(ax, *specs[notec36 + '_clim'][:2], colors[1], lw=1.6, alpha_fill=0.20)
    if notec72:
        fill_spec(ax, *specs[notec72 + '_clim'][:2], colors[2], lw=1.9, alpha_fill=0.22)

    ax.axhline(fap[fk], color='k', ls=':', lw=0.9, alpha=0.6)
    ax.text(9.3, fap[fk] * 1.04, '1% FAP',
            fontsize=6.5, ha='right', va='bottom', color='0.4')

    for per, col, lbl in ref_cl:
        ax.axvline(per, color=col, ls='--', lw=0.9, alpha=0.7)
        ax.text(per * 1.06, ylim[1] * 0.96, lbl,
                fontsize=6.5, color=col, va='top')

    ax.set_xlim(0.5, 10)
    ax.set_ylim(*ylim)
    ax.set_xscale('log')
    ax.set_xlabel('Period (years) — log scale', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f'{x:.0f}' if x >= 1 else f'{x:.1f}'))

leg_cl = [
    Line2D([0],[0], color=G0, lw=2,           label='Raw'),
    Line2D([0],[0], color=G1, lw=2,           label=f'−{MA_SHORT}m MA  (no tectonic)'),
    Line2D([0],[0], color=G2, lw=2,           label=f'−{MA_LONG}m MA  (no tectonic)'),
    Line2D([0],[0], color='k', lw=1, ls=':',  label='1% FAP'),
]
ax_cl13.legend(handles=leg_cl, fontsize=7, frameon=True,
               framealpha=0.85, edgecolor='0.7', loc='upper right')

# ── Panel labels and row annotations ──────────────────────────────────────────
for ax, lbl in zip(
        [ax_ts13, ax_ts18, ax_lp13, ax_lp18, ax_cl13, ax_cl18],
        ['(a)',   '(b)',   '(c)',   '(d)',   '(e)',   '(f)']):
    ax.text(0.012, 0.97, lbl, transform=ax.transAxes,
            fontsize=8.5, va='top', fontweight='bold')

for ax, lbl in [(ax_lp13, 'Tectonic band\n(5–80 yr)'),
                (ax_cl13, 'Climate band\n(0.5–10 yr)')]:
    ax.annotate(lbl, xy=(1.025, 0.5), xycoords='axes fraction',
                fontsize=7.5, ha='left', va='center', color='0.4')

fig.suptitle(
    r'Separating tectonic and climate signals — $\delta^{13}$C and $\delta^{18}$O',
    fontsize=11, y=0.998)

# ── Save ──────────────────────────────────────────────────────────────────────
#plt.savefig(OUT_PDF, bbox_inches='tight', dpi=200)
#plt.savefig(OUT_PNG, bbox_inches='tight', dpi=180)
print(f'Saved  {OUT_PDF}  and  {OUT_PNG}')

fig